# Notebook 11 — Learned Gap Model

**prime-numbers-lab**

This notebook tests whether the reconstruction limit observed in Notebook 10 comes from the gap model itself.

Notebook 10 showed that adaptive weighting does not materially improve reconstruction once density and local-gap constraints are already saturated. Notebook 11 replaces the baseline

`expected gap ≈ log(x)`

with learned corrections:

1. log baseline
2. affine log baseline
3. sinusoidal log-space correction
4. local rolling correction
5. combined learned correction

Goal:

> decide whether prime-gap reconstruction improves when the expected-gap model is learned rather than fixed.

In [ ]:
# --- Standard imports ---
import math
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (11, 6),
    "axes.grid": True,
    "font.size": 12,
})

NOTEBOOK_ID = "11"
SLUG = "learned_gap_model"
RUN_DIR = Path(f"{NOTEBOOK_ID}_{SLUG}")
DATA_DIR = RUN_DIR / "data"
DOCS_DIR = RUN_DIR / "docs"
FIG_DIR = RUN_DIR / "figures"
TEX_DIR = RUN_DIR / "tex"

for d in [RUN_DIR, DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(9423)
print(f"Run directory: {RUN_DIR}")


## 1. Generate primes and gap data

We use a sieve of Eratosthenes so the notebook is self-contained and reproducible in Colab.

In [ ]:
def sieve_primes(n: int) -> np.ndarray:
    """Return all primes <= n."""
    if n < 2:
        return np.array([], dtype=np.int64)
    is_prime = np.ones(n + 1, dtype=bool)
    is_prime[:2] = False
    limit = int(n**0.5) + 1
    for p in range(2, limit):
        if is_prime[p]:
            is_prime[p*p:n+1:p] = False
    return np.flatnonzero(is_prime).astype(np.int64)

MAX_N = 1_000_000
primes = sieve_primes(MAX_N)
xs = primes[:-1].astype(float)
gaps = np.diff(primes).astype(float)
logx = np.log(xs)

# Avoid tiny-x edge effects for model fitting.
mask = xs >= 100
x = xs[mask]
y = gaps[mask]
lx = np.log(x)

print(f"primes <= {MAX_N:,}: {len(primes):,}")
print(f"gap observations after x>=100: {len(y):,}")
print(f"mean gap: {y.mean():.4f}, mean log(x): {lx.mean():.4f}")


## 2. Baseline and learned gap models

The baseline is `log(x)`.  Learned models try to preserve interpretability:

- **affine log**: `a log(x) + b`
- **sinusoidal correction**: `a log(x) + b + c sin(k log x) + d cos(k log x)`
- **local rolling correction**: smoothed residual correction over x-windows
- **combined correction**: sinusoidal model plus local residual smoothing

In [ ]:
def fit_linear_design(X: np.ndarray, y: np.ndarray):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ beta
    return beta, pred

def metrics(y_true, y_pred):
    residual = y_true - y_pred
    mae = float(np.mean(np.abs(residual)))
    rmse = float(np.sqrt(np.mean(residual**2)))
    bias = float(np.mean(residual))
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if np.std(y_pred) > 0 else float("nan")
    return {"MAE": mae, "RMSE": rmse, "bias": bias, "corr": corr}

# Model 0: fixed baseline.
pred_log = lx.copy()

# Model 1: affine log.
X_affine = np.column_stack([lx, np.ones_like(lx)])
beta_affine, pred_affine = fit_linear_design(X_affine, y)

# Model 2: choose a sinusoidal log-frequency by validation-like grid on full diagnostic sample.
freq_grid = np.linspace(0.5, 12.0, 80)
best = None
for k in freq_grid:
    X = np.column_stack([lx, np.ones_like(lx), np.sin(k * lx), np.cos(k * lx)])
    beta, pred = fit_linear_design(X, y)
    rmse = metrics(y, pred)["RMSE"]
    if best is None or rmse < best[0]:
        best = (rmse, k, beta, pred)

_, best_k, beta_sin, pred_sin = best
print(f"affine beta: slope={beta_affine[0]:.6f}, intercept={beta_affine[1]:.6f}")
print(f"best sinusoidal log-frequency k={best_k:.4f}")
print(f"sin beta={beta_sin}")


## 3. Local rolling correction

A purely global model still misses local structure.  We therefore smooth residuals over neighboring observations and add that correction back to the baseline.

This is not a proof of prediction power.  It is a diagnostic: it asks whether the previous limit was caused by using an underfit expected-gap model.

In [ ]:
def rolling_mean_same(values: np.ndarray, window: int) -> np.ndarray:
    """Centered rolling mean with edge padding, same length as input."""
    if window <= 1:
        return values.copy()
    if window % 2 == 0:
        window += 1
    pad = window // 2
    padded = np.pad(values, (pad, pad), mode="edge")
    kernel = np.ones(window) / window
    return np.convolve(padded, kernel, mode="valid")

WINDOW = 401
resid_log = y - pred_log
local_corr = rolling_mean_same(resid_log, WINDOW)
pred_local = pred_log + local_corr

resid_sin = y - pred_sin
local_corr_sin = rolling_mean_same(resid_sin, WINDOW)
pred_combined = pred_sin + local_corr_sin

models = {
    "log_baseline": pred_log,
    "affine_log": pred_affine,
    "sinusoidal_log": pred_sin,
    "local_rolling": pred_local,
    "combined": pred_combined,
}

score_rows = []
for name, pred in models.items():
    row = {"model": name}
    row.update(metrics(y, pred))
    score_rows.append(row)

scores = pd.DataFrame(score_rows).sort_values("RMSE")
scores.to_csv(DATA_DIR / "11_gap_model_scores.csv", index=False)
scores


## 4. Plot model comparison

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(scores["model"], scores["RMSE"])
ax.set_title("Gap model RMSE comparison")
ax.set_xlabel("model")
ax.set_ylabel("RMSE")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(FIG_DIR / "11_gap_model_rmse.png", dpi=180)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(scores["model"], scores["MAE"])
ax.set_title("Gap model MAE comparison")
ax.set_xlabel("model")
ax.set_ylabel("MAE")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(FIG_DIR / "11_gap_model_mae.png", dpi=180)
plt.show()


## 5. Expected gap curves

The plot below compares the fixed `log(x)` curve against learned expected-gap curves.

In [ ]:
# Downsample points for visibility.
idx = np.linspace(0, len(x)-1, 800).astype(int)

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(x[idx], y[idx], s=8, alpha=0.25, label="observed gaps")
ax.plot(x[idx], pred_log[idx], linewidth=2, label="log baseline")
ax.plot(x[idx], pred_affine[idx], linewidth=2, label="affine log")
ax.plot(x[idx], pred_sin[idx], linewidth=2, label="sinusoidal log")
ax.plot(x[idx], pred_combined[idx], linewidth=2, label="combined")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Learned expected-gap curves")
ax.set_xlabel("x")
ax.set_ylabel("gap")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "11_expected_gap_curves.png", dpi=180)
plt.show()


## 6. Residual diagnostics by x-window

This reproduces the Notebook 9/10 logic: if a model is better, residual drift should shrink across windows.

In [ ]:
def window_stats(xvals, ytrue, pred, n_windows=40):
    edges = np.linspace(float(xvals.min()), float(xvals.max()), n_windows + 1)
    rows = []
    for i in range(n_windows):
        lo, hi = edges[i], edges[i+1]
        m = (xvals >= lo) & (xvals < hi if i < n_windows-1 else xvals <= hi)
        if m.sum() < 10:
            continue
        residual = ytrue[m] - pred[m]
        expected = pred[m]
        rows.append({
            "window": i,
            "midpoint": 0.5 * (lo + hi),
            "count": int(m.sum()),
            "mean_gap": float(ytrue[m].mean()),
            "mean_expected": float(expected.mean()),
            "mean_residual": float(residual.mean()),
            "mae": float(np.mean(np.abs(residual))),
            "rmse": float(np.sqrt(np.mean(residual**2))),
            "relative_drift": float(np.mean(np.abs(residual)) / max(np.mean(expected), 1e-12)),
        })
    return pd.DataFrame(rows)

window_frames = []
for name, pred in models.items():
    dfw = window_stats(x, y, pred)
    dfw["model"] = name
    window_frames.append(dfw)
window_df = pd.concat(window_frames, ignore_index=True)
window_df.to_csv(DATA_DIR / "11_window_residual_diagnostics.csv", index=False)
window_df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for name in models:
    sub = window_df[window_df["model"] == name]
    ax.plot(sub["midpoint"], sub["relative_drift"], marker="o", label=name)
ax.set_xscale("log")
ax.set_title("Windowed relative residual drift by model")
ax.set_xlabel("window midpoint")
ax.set_ylabel("relative residual drift")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "11_window_relative_residual_drift.png", dpi=180)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for name in models:
    sub = window_df[window_df["model"] == name]
    ax.plot(sub["midpoint"], sub["mean_residual"], marker="o", label=name)
ax.axhline(0, linestyle="--")
ax.set_xscale("log")
ax.set_title("Windowed mean residual by model")
ax.set_xlabel("window midpoint")
ax.set_ylabel("mean residual")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "11_window_mean_residual.png", dpi=180)
plt.show()


## 7. Improvement ratios

We compare each learned model to the fixed `log(x)` baseline.

In [ ]:
base_rmse = float(scores.loc[scores["model"] == "log_baseline", "RMSE"].iloc[0])
base_mae = float(scores.loc[scores["model"] == "log_baseline", "MAE"].iloc[0])

improvement = scores.copy()
improvement["RMSE_improvement_vs_log"] = 1 - improvement["RMSE"] / base_rmse
improvement["MAE_improvement_vs_log"] = 1 - improvement["MAE"] / base_mae
improvement.to_csv(DATA_DIR / "11_gap_model_improvement.csv", index=False)
improvement


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
plot_df = improvement.sort_values("RMSE_improvement_vs_log", ascending=False)
ax.bar(plot_df["model"], plot_df["RMSE_improvement_vs_log"])
ax.axhline(0, linestyle="--")
ax.set_title("RMSE improvement versus log(x) baseline")
ax.set_xlabel("model")
ax.set_ylabel("improvement share")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(FIG_DIR / "11_rmse_improvement_vs_log.png", dpi=180)
plt.show()


## 8. Interpretation document

Write a concise interpretation with figure links for GitHub browsing.

In [ ]:
best_model = scores.iloc[0]["model"]
best_rmse = float(scores.iloc[0]["RMSE"])
log_rmse = float(scores.loc[scores["model"] == "log_baseline", "RMSE"].iloc[0])
best_improvement = 1 - best_rmse / log_rmse

interpretation = f"""# Learned Gap Model (Notebook 11)

## Purpose

Notebook 10 showed that adaptive weighting did not materially improve reconstruction once density and local-gap constraints were saturated.

Notebook 11 tests whether the remaining limit comes from the expected-gap model itself.

Baseline model:

```text
expected gap ≈ log(x)
```

Learned alternatives:

- affine log model
- sinusoidal log-space correction
- local rolling residual correction
- combined sinusoidal + local correction

---

## 1. Model score comparison

![Gap model RMSE](../figures/11_gap_model_rmse.png)

![Gap model MAE](../figures/11_gap_model_mae.png)

Best model: `{best_model}`

Best RMSE: `{best_rmse:.6f}`

Baseline log RMSE: `{log_rmse:.6f}`

RMSE improvement vs log baseline: `{best_improvement:.4%}`

Interpretation:

If improvement is small, the fixed log model is already close to the available signal at this scale. If improvement is substantial, the Notebook 10 ceiling was model-limited rather than weighting-limited.

---

## 2. Expected-gap curves

![Expected gap curves](../figures/11_expected_gap_curves.png)

This plot compares observed prime gaps to expected-gap curves.

The main question is whether learned curves track local structure better than `log(x)` without simply overfitting pointwise noise.

---

## 3. Windowed residual drift

![Windowed relative residual drift](../figures/11_window_relative_residual_drift.png)

The key diagnostic is windowed relative residual drift.

A better expected-gap model should reduce drift across windows, not only improve global RMSE.

---

## 4. Windowed mean residual

![Windowed mean residual](../figures/11_window_mean_residual.png)

A stable model keeps mean residuals closer to zero across scale.

---

## 5. Improvement ratio

![RMSE improvement vs log](../figures/11_rmse_improvement_vs_log.png)

This compares each learned model against the fixed `log(x)` baseline.

---

## Core result

Notebook 11 distinguishes two possibilities:

1. **Small improvement**: reconstruction has reached a structural limit for these features.
2. **Large improvement**: expected-gap modeling was the missing piece after Notebook 10.

In either case, Notebook 11 converts the Notebook 10 observation into a sharper diagnostic:

> adaptation is not enough unless expected-gap structure improves.

---

Constraint → signal > noise
"""

(DOCS_DIR / "interpretation.md").write_text(interpretation, encoding="utf-8")
print(interpretation[:1200])


## 9. Export package

In [ ]:
summary = {
    "notebook": "11_learned_gap_model",
    "max_n": MAX_N,
    "observations": int(len(y)),
    "best_model": str(best_model),
    "best_rmse": best_rmse,
    "log_baseline_rmse": log_rmse,
    "best_improvement_vs_log": best_improvement,
    "best_sinusoidal_frequency": float(best_k),
    "rolling_window": WINDOW,
}

(DATA_DIR / "11_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

EXPORT_NAME = f"{NOTEBOOK_ID}_{SLUG}_export.zip"
with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)
